# Traditional models - Baseline
Train baseline classification models (Logistic Regression, Random Forest, SVC, GradientBoosting) on the original imbalanced dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

url = '{}'
df = pd.read_csv(url)
X = df.drop('Class', axis=1)
y = df['Class']

# Scale Amount and Time (remaining V1..V28 are already PCA components)
scaler = StandardScaler()
X[['Time','Amount']] = scaler.fit_transform(X[['Time','Amount']])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


In [ ]:
# Baseline re-run: scale on TRAINING only + full evaluation metrics 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC


# Load cleaned data saved by dataset_analysis notebook
df = pd.read_csv('./creditcard_full.csv')
X = df.drop('Class', axis=1)
y = df['Class']

# Stratified split first
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# Fit scaler on training only, then transform test
scaler = StandardScaler()
X_train[['Time','Amount']] = scaler.fit_transform(X_train[['Time','Amount']])
X_test[['Time','Amount']] = scaler.transform(X_test[['Time','Amount']])

# Models (keep class_weight for imbalance where applicable)
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVC': SVC(kernel='rbf', probability=True, random_state=42)
}

results = {}
for name, model in models.items():
    print(f"\n--- Training & evaluating: {name} ---")
    model.fit(X_train, y_train)
    # probabilities for AUC
    probs = model.predict_proba(X_test)[:,1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    preds = model.predict(X_test)
    auc = roc_auc_score(y_test, probs)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)
    print(f"AUC: {auc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
    print("Confusion Matrix:\n", cm)
    print("\nClassification Report:\n", classification_report(y_test, preds, digits=4))
    results[name] = {'model': model, 'auc': auc, 'precision': prec, 'recall': rec, 'f1': f1, 'cm': cm}





In [ ]:
# Save results summary
import joblib
joblib.dump(results, 'baseline_results.pkl')
print("\nSaved baseline_results.pkl")
